In [1]:
!pip install sentence_transformers

In [2]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install contractions

Note: you may need to restart the kernel to use updated packages.


In [4]:
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from contractions import fix
import warnings
warnings.simplefilter('ignore')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [5]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/jupyter/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
import pandas as pd
import re

In [7]:
df = pd.read_parquet('english_reviews_output.parquet')

In [8]:
df.shape

(499456, 54)

In [9]:
df.columns

Index(['business_id', 'business_category', 'business_name', 'city_x',
       'state_x', 'business_star_rating', 'review_count_x', 'is_open_x',
       'ByAppointmentOnly', 'BusinessAcceptsCreditCards', 'CoatCheck',
       'RestaurantsTakeOut', 'RestaurantsDelivery', 'Caters', 'WiFi',
       'WheelchairAccessible', 'HappyHour', 'OutdoorSeating', 'HasTV',
       'RestaurantsReservations', 'DogsAllowed', 'Alcohol', 'GoodForKids',
       'RestaurantsTableService', 'RestaurantsGoodForGroups', 'DriveThru',
       'NoiseLevel', 'Smoking', 'Corkage', 'MusicCategory', 'AmbienceCategory',
       'parking_final', 'text', 'reviews_received_cool', 'reviews_star_rating',
       'date', 'reviews_received_funny', 'review_id',
       'reviews_received_useful', 'user_id', 'review_year', 'name',
       'review_count', 'yelping_since', 'user_sent_useful', 'user_sent_funny',
       'user_sent_cool', 'elite', 'average_stars', 'yelping_end_year',
       'yelping_start_year', 'elite_member', 'target', 'languag

In [10]:
def clean_text(text):
    """
    Clean and preprocess app descriptions
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Expand contractions
    text = fix(text)
    
    return text

def remove_stopwords(text):
    """
    Remove stopwords from text
    """
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

In [11]:
# Perform random sampling of 250,000 rows
sampled_df = df.sample(n=250000, random_state=42)

In [12]:
# Clean descriptions
sampled_df['cleaned_desc'] = sampled_df['text'].apply(clean_text)

In [13]:
sampled_df['cleaned_desc'] = sampled_df['text'].apply(remove_stopwords)

In [14]:
sampled_df.shape

(250000, 55)

In [15]:
sampled_df = sampled_df.dropna(subset=['target']).reset_index(drop=True)

In [16]:
sampled_df.shape

(225629, 55)

In [17]:
sampled_df['target'].value_counts()

target
1.0    173320
0.0     52309
Name: count, dtype: int64

In [18]:
sampled_df.columns

Index(['business_id', 'business_category', 'business_name', 'city_x',
       'state_x', 'business_star_rating', 'review_count_x', 'is_open_x',
       'ByAppointmentOnly', 'BusinessAcceptsCreditCards', 'CoatCheck',
       'RestaurantsTakeOut', 'RestaurantsDelivery', 'Caters', 'WiFi',
       'WheelchairAccessible', 'HappyHour', 'OutdoorSeating', 'HasTV',
       'RestaurantsReservations', 'DogsAllowed', 'Alcohol', 'GoodForKids',
       'RestaurantsTableService', 'RestaurantsGoodForGroups', 'DriveThru',
       'NoiseLevel', 'Smoking', 'Corkage', 'MusicCategory', 'AmbienceCategory',
       'parking_final', 'text', 'reviews_received_cool', 'reviews_star_rating',
       'date', 'reviews_received_funny', 'review_id',
       'reviews_received_useful', 'user_id', 'review_year', 'name',
       'review_count', 'yelping_since', 'user_sent_useful', 'user_sent_funny',
       'user_sent_cool', 'elite', 'average_stars', 'yelping_end_year',
       'yelping_start_year', 'elite_member', 'target', 'languag

In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

In [20]:
sampled_df = sampled_df[['review_id','review_year','elite_member','business_id','user_id','elite','cleaned_desc','reviews_star_rating']]

In [21]:
# 3. Generate BERT embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(sampled_df['cleaned_desc'].tolist(), show_progress_bar=True, batch_size=64)

Batches:   0%|          | 0/3526 [00:00<?, ?it/s]

In [22]:
# 4. KMeans clustering (binary)
kmeans = KMeans(n_clusters=2, random_state=42)
sampled_df['cluster'] = kmeans.fit_predict(embeddings)

In [23]:
# 5. (Optional) Evaluate using stars as proxy labels
def map_star_to_sentiment(star):
    if star >= 4:
        return 1
    elif star <= 2:
        return 0
    else:
        return -1

In [24]:
sampled_df['true_sentiment'] = sampled_df['reviews_star_rating'].apply(map_star_to_sentiment)

In [25]:
df_eval = sampled_df[sampled_df['true_sentiment'] != -1]

In [26]:
df_eval['pred_sentiment'] = df_eval['cluster']  # might need to flip later

In [32]:
acc = accuracy_score(df_eval['true_sentiment'], df_eval['pred_sentiment'])

In [33]:
print(f"BERT-based unsupervised accuracy (vs. proxy): {acc:.4f}")

BERT-based unsupervised accuracy (vs. proxy): 0.87


In [29]:
sampled_df['cluster'].value_counts()

cluster
1    166464
0     59165
Name: count, dtype: int64

In [30]:
## Cluster 0 is positive, cluster 1 is negative
df_eval['flipped_pred'] = 1 - df_eval['pred_sentiment']
flipped_acc = accuracy_score(df_eval['true_sentiment'], df_eval['flipped_pred'])

In [31]:
flipped_acc

0.13344915768806315

In [34]:
from sklearn.metrics import silhouette_score

score = silhouette_score(embeddings, sampled_df['cluster'])
print(f'Silhouette Score: {score:.3f}')

Silhouette Score: 0.058


In [35]:
df_eval.columns

Index(['review_id', 'review_year', 'elite_member', 'business_id', 'user_id',
       'elite', 'cleaned_desc', 'reviews_star_rating', 'cluster',
       'true_sentiment', 'pred_sentiment', 'flipped_pred'],
      dtype='object')

In [36]:
# Assuming your DataFrame is named 'df'
df_eval.to_parquet('Question2_250k_output_with_clusters.parquet', index=False)